# 02 — Data preparation

This notebook prepares the analytical cohort used throughout the project and the predictor sets required for RQ1.

The broader variable audit and candidate selection were performed in '01-data-audit.ipynb'. Here, the HICP target and analytical cohort are constructed, RQ1 predictors are prepared, special response codes are handled, and the model-ready dataset is defined.

Pain-management variables used in RQ2 and RQ3 are prepared separately in the corresponding analysis notebook.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.hicp import (
    flag_chronic_pain,
    flag_hicp,
    flag_cohort,
    recode_invalid_to_nan,
    PAIN_INVALID_CODES,
)

df = pd.read_csv("../data/raw/adult25csv/adult25.csv", low_memory=False)

## Target and analytical cohort

### Building the HICP target

In [2]:
df["chronic_pain"] = flag_chronic_pain(df)
df["hicp"] = flag_hicp(df)

In [3]:
df[["PAIFRQ3M_A", "PAIWKLM3M_A", "chronic_pain", "hicp"]].head(10)

,PAIFRQ3M_A,PAIWKLM3M_A,chronic_pain,hicp
0,4,4.0,True,True
1,2,1.0,False,False
2,1,NaN,False,False
3,2,2.0,False,False
4,2,1.0,False,False
5,1,NaN,False,False
6,3,2.0,True,False
7,1,NaN,False,False
8,2,1.0,False,False
9,2,2.0,False,False


### Defining the analytical cohort

In [4]:
cohort = flag_cohort(df)

In [5]:
n_invalid = df.loc[df["chronic_pain"], "PAIWKLM3M_A"].isin(PAIN_INVALID_CODES).sum()

print(f"Refused / not ascertained / don't know within chronic pain: {n_invalid}")

Refused / not ascertained / don't know within chronic pain: 9


-> The chronic-pain group contains 5601 respondents.

-> Nine respondents with chronic pain gave a special response code to the pain-limitation question and therefore cannot be classified as either HICP or non-HICP.

-> They are excluded from the analytical cohort; this gives us a total of **5592 respondents with chronic pain and known limitation status**.

### Target distribution

In [6]:
analysis_df = df.loc[cohort].copy()

print("Analytical cohort shape:", analysis_df.shape)
print("\nHICP distribution:")
print(analysis_df["hicp"].value_counts())

Analytical cohort shape: (5592, 579)

HICP distribution:
hicp
False    3636
True     1956
Name: count, dtype: int64


In [7]:
print(f"\nHICP share: {analysis_df['hicp'].mean() * 100:.1f}%")


HICP share: 35.0%


The final analytical cohort contains 5592 adults with chronic pain and known limitation status:

- 1956 meet the definition of HICP.
- 3636 have chronic pain but do not meet the HICP definition.
- HICP therefore represents 35% of the analytical cohort.

The cohort definition only requires a valid chronic-pain classification and a known limitation status; respondents may still have missing or special responses for individual predictors.

-----

## Pain intensity recoding

**PAIAMNT_A** records pain intensity, but its raw response codes are not ordered by intensity:

- 1 = 'a little'
- 2 = 'a lot'
- 3 = 'somewhere in between'

Because pain intensity is used as an ordered predictor in RQ1, the raw numeric codes cannot be used directly. A new variable, **pain_intensity_ord** is therefore created so that increasing values correspond to increasing pain intensity:

- 1 = 'a little'
- 2 = 'somewhere in between'
- 3 = 'a lot'

The original NHIS variable is retained unchanged.

In [8]:
analysis_df["pain_intensity_ord"] = analysis_df["PAIAMNT_A"].map({
    1: 1,  # a little
    3: 2,  # somewhere in between
    2: 3   # a lot
})

In [9]:
pd.crosstab(analysis_df["PAIAMNT_A"], analysis_df["pain_intensity_ord"])

pain_intensity_ord,1.0,2.0,3.0
PAIAMNT_A,,,
1.0,1122,0,0
2.0,0,0,1800
3.0,0,2669,0


In [10]:
analysis_df.loc[analysis_df["pain_intensity_ord"].isna(),["PAIAMNT_A", "PAIFRQ3M_A", "PAIWKLM3M_A", "hicp"]]

,PAIAMNT_A,PAIFRQ3M_A,PAIWKLM3M_A,hicp
13454,7.0,3,2.0,False


**Check:** 

The analytical cohort contains 5592 respondents, but only 5591 have a valid pain-intensity response.

One respondent with chronic pain and known limitation status has a special response code, not valid response (**PAIAMNT_A** = 7) for pain intensity. This respondent remains part of the analytical cohort but cannot contribute to analyses requiring pain intensity.

Special response codes are recoded to missing values before modelling.

-----

## Predictor sets for RQ1

Predictor sets were fixed before model fitting, following the modelling design documented in `docs/business-case.md`.

### Model 0 — pain intensity only

Model 0 uses only the recoded pain-intensity variable. 

It provides the benchmark for the second part of RQ1: 
> whether pain intensity alone is sufficient to distinguish HICP from non-HICP chronic pain.

In [11]:
model_0_features = ["pain_intensity_ord"]   #PAIAMNT_A = pain intensity only 

### Model A — routine consultation information

Model A extends pain intensity with information that would normally already be available in a short clinical assessment: age, sex, and reported pain location.

- `AGEP_A`: age of the sample adult
- `SEX_A`: sex of the sample adult
- `PAIBACK3M_A`: back pain
- `PAIULMB3M_A`: pain in hands, arms or shoulders
- `PAILLMB3M_A`: pain in hips, knees or feet
- `PAIHDFC3M_A`: headache or migraine
- `PAIAPG3M_A`: abdominal, pelvic or genital pain
- `PAITOOTH3M_A`: toothache or jaw pain

All six pain-location variables are retained rather than selecting locations based on their relationship with HICP. This keeps the baseline specification independent of the outcome.

In [12]:
pain_location_features = [
    "PAIBACK3M_A",
    "PAIULMB3M_A",
    "PAILLMB3M_A",
    "PAIHDFC3M_A",
    "PAIAPG3M_A",
    "PAITOOTH3M_A",
]

model_a_features = [
    "pain_intensity_ord",
    "AGEP_A",
    "SEX_A",
    *pain_location_features  # insert each pain-location variable into the feature list
]

### Model B — biopsychosocial layer

Model B adds the eight biopsychosocial variables fixed in the business case before model fitting.

These variables were selected from the literature-informed candidate set audited in '01-data-audit.ipynb'. Selection was based on conceptual coverage, redundancy, data quality, and cross-sectional interpretability, but ***not on their observed association with HICP***.

The final layer covers psychological symptoms, sleep, coping, social support, socioeconomic position, health behaviour, and comorbidity.

In [13]:
biopsychosocial_features = [
    "PHQCAT_A",
    "GADCAT_A",
    "WPHSLEEP_A",
    "WPHSTRESS_A",
    "SUPPORT_A",
    "EDUCP_A",
    "SMKCIGST_A",
    "ARTHEV_A",
]

model_b_features = model_a_features + biopsychosocial_features

In [14]:
print("Model 0:", model_0_features)
print("Model A:", model_a_features)
print("Model B:", model_b_features)

Model 0: ['pain_intensity_ord']
Model A: ['pain_intensity_ord', 'AGEP_A', 'SEX_A', 'PAIBACK3M_A', 'PAIULMB3M_A', 'PAILLMB3M_A', 'PAIHDFC3M_A', 'PAIAPG3M_A', 'PAITOOTH3M_A']
Model B: ['pain_intensity_ord', 'AGEP_A', 'SEX_A', 'PAIBACK3M_A', 'PAIULMB3M_A', 'PAILLMB3M_A', 'PAIHDFC3M_A', 'PAIAPG3M_A', 'PAITOOTH3M_A', 'PHQCAT_A', 'GADCAT_A', 'WPHSLEEP_A', 'WPHSTRESS_A', 'SUPPORT_A', 'EDUCP_A', 'SMKCIGST_A', 'ARTHEV_A']


-----

## RQ1 predictor quality checks

Model B contains every predictor used in Models 0 and A, plus the biopsychosocial layer. Therefore, it represents the full set of variables used anywhere in RQ1.

The checks below assess coding, special response values, and missingness before the modelling dataset is finalized.

In [15]:
all_rq1_features = model_b_features.copy()

# Model B contains all predictors used in Models 0 and A
# so this is the full predictor set used across RQ1.

### Special response codes

NHIS uses numeric codes for responses such as refusal, 'not ascertained', and 'don't know'. These values are stored as numbers in the raw dataset, so pandas does not automatically recognize them as missing.

Before modelling, these special response codes are explicitly identified and converted to missing values ('NaN').

In [16]:
rq1_invalid_codes = {
    "pain_intensity_ord": [],
    "AGEP_A": [97, 98, 99],
    "SEX_A": [7, 8, 9],

    "PAIBACK3M_A": [7, 8, 9],
    "PAIULMB3M_A": [7, 8, 9],
    "PAILLMB3M_A": [7, 8, 9],
    "PAIHDFC3M_A": [7, 8, 9],
    "PAIAPG3M_A": [7, 8, 9],
    "PAITOOTH3M_A": [7, 8, 9],

    "PHQCAT_A": [8],
    "GADCAT_A": [8],
    "WPHSLEEP_A": [7, 8, 9],
    "WPHSTRESS_A": [7, 8, 9],
    "SUPPORT_A": [7, 8, 9],
    "EDUCP_A": [97, 98, 99],
    "SMKCIGST_A": [9],
    "ARTHEV_A": [7, 8, 9],
}

-> The following check counts **how many special response codes are present in each RQ1 predictor**.

-> This provides a **compact summary of invalid responses** before they are recoded to missing values ('NaN') and avoids printing the full distribution of every variable.

In [17]:
special_counts = pd.Series({
    col: analysis_df[col].isin(codes).sum()
    for col, codes in rq1_invalid_codes.items()
    if codes
}).sort_values(ascending=False)

special_counts[special_counts > 0]

SUPPORT_A       52
SMKCIGST_A      27
GADCAT_A        21
EDUCP_A         18
PHQCAT_A        16
PAIAPG3M_A      12
PAIHDFC3M_A     11
WPHSTRESS_A     11
ARTHEV_A        11
PAITOOTH3M_A    10
AGEP_A           8
SEX_A            7
PAIULMB3M_A      7
PAILLMB3M_A      7
PAIBACK3M_A      4
WPHSLEEP_A       4
dtype: int64

> The special response codes identified above **do not represent substantive answers and must not be treated as predictor categories**.

So they are converted to missing values ('NaN'), and the original valid response codes are left unchanged.

In [18]:
for col, codes in rq1_invalid_codes.items():  #for each variable -> look at the invalid codes
    if codes:
        analysis_df[col] = analysis_df[col].mask(analysis_df[col].isin(codes)) # replace special response codes with NaN

### Missingness after recoding

After converting special response codes to 'NaN', missing values are reassessed across all predictors used in RQ1.

This shows the true amount of unusable predictor data before defining the final model-ready sample.

In [19]:
missing_after_recode = (analysis_df[all_rq1_features].isna().sum().sort_values(ascending=False))

missing_after_recode[missing_after_recode > 0]

SUPPORT_A             52
SMKCIGST_A            27
GADCAT_A              21
EDUCP_A               18
PHQCAT_A              16
PAIAPG3M_A            12
ARTHEV_A              11
WPHSTRESS_A           11
PAIHDFC3M_A           11
PAITOOTH3M_A          10
AGEP_A                 8
PAILLMB3M_A            7
PAIULMB3M_A            7
SEX_A                  7
WPHSLEEP_A             4
PAIBACK3M_A            4
pain_intensity_ord     1
dtype: int64

> **The counts above now represent missing values rather than special response codes.**

-> Because the same respondent may have missing values in more than one predictor, these counts cannot be added together to determine how many respondents will be excluded from modelling. The next step therefore defines the complete-case modelling sample at respondent level.

### Final model-ready sample

To compare Models 0, A and B fairly, the same complete-case sample is used across all three models.

Respondents with a missing value in any predictor required by Model B are excluded from the modelling sample.

In [20]:
# Keep the same sample for all three models
model_ready_df = analysis_df.dropna(subset=model_b_features).copy()

In [21]:
# Check how many respondents are left
print("Analytical cohort:", len(analysis_df))
print("Model-ready cohort:", len(model_ready_df))
print("Excluded due to missing predictors:",len(analysis_df) - len(model_ready_df))

Analytical cohort: 5592
Model-ready cohort: 5465
Excluded due to missing predictors: 127


In [22]:
# Check whether the HICP distribution changed
print("\nHICP distribution:")
print(model_ready_df["hicp"].value_counts())
print(f"\nHICP share: {model_ready_df['hicp'].mean() * 100:.1f}%")


HICP distribution:
hicp
False    3557
True     1908
Name: count, dtype: int64

HICP share: 34.9%


##### The final model-ready sample contains 5465 respondents, with 127 excluded due to missing predictor values.

> The HICP proportion remains essentially unchanged (34.9% vs. 35% in the analytical cohort), suggesting that the complete-case restriction did not materially alter the target distribution.

------

## Prepared analytical dataset

The **final RQ1 dataset** contains the **HICP target, all predictors used across Models 0, A and B, and the NHIS survey-design variables** that may be required in later analyses.

Only respondents in the complete-case modelling sample are retained.

In [23]:
# Keep only the variables needed for RQ1
rq1_df = model_ready_df[
    [
        "hicp",   # target
        *all_rq1_features,  # predictors used across Models 0, A and B

        # NHIS survey-design variables
        "WTFA_A",
        "PSTRAT",
        "PPSU",
    ]
].copy()

In [24]:
rq1_df.shape

(5465, 21)

In [25]:
# Save the prepared RQ1 dataset
rq1_df.to_csv("../data/processed/rq1_model_ready.csv",index=False)

In [26]:
# Confirm the saved dataset
pd.read_csv("../data/processed/rq1_model_ready.csv").shape

(5465, 21)

#### To conclude:

The prepared RQ1 dataset was saved to 'data/processed/rq1_model_ready.csv'.

> It contains 5465 respondents and 21 variables, including the HICP target, all predictors required for Models 0, A and B, and the NHIS survey-design variables.

-> This dataset will be used as the starting point for the RQ1 exploratory analysis and modelling notebooks.